# Phase 3 — Reasoning Finetuning, Evaluation and Attention Analysis

Finetunes **both** models on their own synthetic reasoning data, scores pretrained
against finetuned, and re-runs the attention toolkit on the finetuned checkpoints.

Unlike Phase 2, this fits comfortably in **one session** — roughly 20 minutes of GPU per
language — so both languages run here and produce a single tarball.

The two models stay completely independent: separate reasoning corpora, separate
tokenizers, separate weights. Model L is never initialised from Model H.

**Before running**, in the right-hand panel:

- **Accelerator → GPU T4 x2** (do this first; changing it restarts the session)
- **Internet → On** (cell 8 pip-installs `sentencepiece` and a Devanagari font)

### Inputs this notebook expects

Per language, in a dataset whose title contains `hindi` or `nepal` (never both):

| File | From the repo |
|---|---|
| `best.pt` | `checkpoints/<lang>/` — the **pretrained** Phase 2 model |
| `train.jsonl`, `validation.jsonl`, `test.jsonl` | `<lang>/reasoning/` |
| `hi.model` / `ne.model` | `<lang>/tokenizer/` |
| `validation.bin`, `validation.meta.json` | `<lang>/data/tokens/` |

Plus the three code datasets: `scripts/`, `common/`, `lma/`.

> `scripts/` and `common/` both gained files for Phase 3 — re-upload them and **refresh
> the version in the Data panel**, or the run uses stale code.

## 1. Control panel

In [ ]:
LANGUAGES     = ["hi", "ne"]   # both fit in one session; set to ["hi"] to do one

RUN_FINETUNE  = True      # the finetuning runs
RUN_EVAL      = True      # pretrained vs finetuned exact match
RUN_ATTENTION = True      # attention heatmaps and per-head statistics

EPOCHS        = 1         # 375 steps. Three epochs overfit hard -- best validation
                          # landed at step 100 of 1125 -- and max_steps also sets the
                          # cosine decay shape, so the learning rate sat at its peak
                          # straight through the overfitting window. One epoch decays
                          # while the model is still improving.
WARMUP_STEPS  = 40        # ~10% of 375
EVAL_EVERY    = 25        # 15 evaluations, enough to pin the minimum
EVAL_BATCHES  = 100       # 3,000 validation examples / 32 = 94 batches, so this covers
                          # the whole split -- and it matches the value the checkpoint
                          # records, which the CLI default of 20 did not
BATCH_SIZE    = 64        # examples per optimiser step
MICRO_BATCH   = 32        # examples per forward pass; drop to 16 or 8 on a CUDA OOM.
                          # Gradient accumulation compensates, so the effective batch
                          # and the learning dynamics are unchanged.
LEARNING_RATE = 1e-4      # an order below pretraining's 6e-4: the model already speaks
                          # the language, this only bends it towards the task
MAX_LEN       = 128       # longest example kept, in tokens (measured max is 59)

EVAL_LIMIT    = None      # None = the full 3,000-example test split
MAX_NEW       = 8         # generation budget per answer

assert all(l in ("hi", "ne") for l in LANGUAGES)
LANG_DIRS = {"hi": "hindi", "ne": "nepali"}
LABELS = {"hi": "Model H (Hindi, higher-resource)",
          "ne": "Model L (Nepali, lower-resource)"}
print("configured for:", ", ".join(LABELS[l] for l in LANGUAGES))

## 2. Environment

In [ ]:
import os, sys, json, time, shutil, subprocess
from pathlib import Path
import torch

print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"  {props.name}  {props.total_memory/1e9:.1f} GB")
else:
    print("\n  !! NO GPU ATTACHED !!")
    print("  Settings -> Accelerator -> GPU T4 x2, then re-run.")

print("\nworking dir space:")
print(subprocess.run(["df", "-h", "/kaggle/working"], capture_output=True, text=True).stdout)

## 3. Assemble the project tree

Inputs are found by **searching** `/kaggle/input` recursively, not by fixed paths, so
dataset names do not matter — with one rule: a language's files must have `hindi` or
`nepal` in their path and never both. Code packages are identified by the modules they
contain, so those can be named anything at all.

In [ ]:
INPUT = Path("/kaggle/input")
ROOT  = Path("/kaggle/working/vidhi")


def link(src: Path, dst: Path) -> None:
    """Symlink one input file into the working tree, replacing any previous link."""
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    dst.symlink_to(src)


def for_language(pattern: str, lang: str) -> list:
    """Find input files matching a pattern whose path names the given language."""
    stem = lang[:5].lower()
    return sorted(p for p in INPUT.rglob(pattern) if stem in str(p).lower())


missing = []
resolved = {"hindi": {}, "nepali": {}}   # language -> {destination: source}

for lang, code_ in (("hindi", "hi"), ("nepali", "ne")):
    if code_ not in LANGUAGES:
        continue

    # The pretrained Phase 2 checkpoint. This is Phase 3's key input -- in Phase 2 it
    # was the output. Both languages have a file called best.pt, so the language
    # substring in the path is what tells them apart.
    hits = for_language("best.pt", lang)
    if hits:
        link(hits[0], ROOT/"checkpoints"/lang/"best.pt")
        resolved[lang][f"checkpoints/{lang}/best.pt"] = hits[0]
    else:
        missing.append(f"checkpoints/{lang}/best.pt (pretrained model)")

    for split in ("train", "validation", "test"):
        hits = for_language(f"{split}.jsonl", lang)
        if hits:
            link(hits[0], ROOT/lang/"reasoning"/f"{split}.jsonl")
            resolved[lang][f"{lang}/reasoning/{split}.jsonl"] = hits[0]
        else:
            missing.append(f"{lang}/reasoning/{split}.jsonl")

    hits = list(INPUT.rglob(f"{code_}.model"))
    if hits:
        link(hits[0], ROOT/lang/"tokenizer"/f"{code_}.model")
        resolved[lang][f"{lang}/tokenizer/{code_}.model"] = hits[0]
    else:
        missing.append(f"{lang}/tokenizer/{code_}.model")

    # Only the validation split is needed here: the per-head attention statistics are
    # computed on it so they stay comparable with the Phase 2 numbers. train.bin is
    # 1.5 GB and Phase 3 never touches it.
    for ext in ("bin", "meta.json"):
        hits = for_language(f"validation.{ext}", lang)
        if hits:
            link(hits[0], ROOT/lang/"data"/"tokens"/f"validation.{ext}")
            resolved[lang][f"{lang}/data/tokens/validation.{ext}"] = hits[0]
        elif RUN_ATTENTION:
            missing.append(f"{lang}/data/tokens/validation.{ext}")

# Files are routed by whether "hindi" or "nepal" appears in the path. A single dataset
# named so that BOTH appear -- "hindi-nepali-reasoning", say -- makes every path match
# both languages, and sorted()[0] would hand Nepali the Hindi checkpoint. Both models
# have identical architecture and vocabulary size, so nothing downstream could catch
# it: Model L would be finetuned from Model H's weights with no visible symptom.
shared = set(resolved["hindi"].values()) & set(resolved["nepali"].values())
if shared:
    raise SystemExit(
        "LANGUAGE CONTAMINATION: the same source file was selected for both models:\n  "
        + "\n  ".join(sorted(str(p) for p in shared))
        + "\n\nCause: a dataset path contains both 'hindi' and 'nepal'. Rename the "
          "Kaggle dataset, or move each language into its own dataset, so that any "
          "one path names exactly one language."
    )

print("resolved inputs:")
for lang in ("hindi", "nepali"):
    for dst, source in resolved[lang].items():
        print(f"  {dst:<44} <- {source}")
print()

# Code datasets -> package directories, identified by the modules they contain rather
# than by dataset name. scripts/ and common/ both contain a clean.py, which is why the
# three must stay separate uploads and why the signatures below avoid that filename.
PACKAGES = {
    "scripts": {"finetune.py", "eval_reasoning.py", "attention_analysis.py"},
    "common":  {"reasoning.py", "normalize.py", "schema.py"},
    "lma":     {"model.py", "checkpoint.py", "schedule.py"},
}


def find_package(signature: set) -> Path | None:
    """Return the input directory containing every file in `signature`, else None.

    Raises if more than one directory matches, which happens when a stale dataset is
    still attached beside a newly uploaded one. Silently taking the first would run
    whichever code sorted earlier -- invisible in the output, and a wasted run.
    """
    matches = [d for d in sorted({f.parent for f in INPUT.rglob("*.py")})
               if signature <= {f.name for f in d.glob("*.py")}]
    if len(matches) > 1:
        raise SystemExit(
            "AMBIGUOUS CODE INPUT: more than one attached dataset provides "
            f"{', '.join(sorted(signature))}:\n  "
            + "\n  ".join(str(m) for m in matches)
            + "\n\nDetach the stale one in the Data panel."
        )
    return matches[0] if matches else None


for package, signature in PACKAGES.items():
    source = find_package(signature)
    if source is None:
        missing.append(f"code for {package}/ "
                       f"(no input directory holds {', '.join(sorted(signature))})")
        continue
    (ROOT/package).mkdir(parents=True, exist_ok=True)
    for f in source.glob("*.py"):
        shutil.copy2(f, ROOT/package/f.name)
    print(f"  {package}/  {len(list((ROOT/package).glob('*.py'))):>2} modules  <- {source}")

if missing:
    raise SystemExit("MISSING INPUTS:\n  " + "\n  ".join(missing))

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print(f"\nproject tree ready at {ROOT}")

## 4. Dependencies

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentencepiece"], check=True)
import sentencepiece
print("sentencepiece", sentencepiece.__version__)

# A Devanagari-capable font, or matplotlib draws every Hindi/Nepali glyph as an empty
# box and the attention heatmaps lose the reasoning prompt they are meant to show.
from matplotlib import font_manager
import matplotlib, pathlib

DEVANAGARI = ("Noto Sans Devanagari", "Noto Serif Devanagari", "Lohit Devanagari",
              "Nirmala UI", "Samyak Devanagari", "Kalimati", "FreeSerif")

def devanagari_font():
    """Return the first installed font that can render Devanagari, else None."""
    available = {f.name for f in font_manager.fontManager.ttflist}
    return next((n for n in DEVANAGARI if n in available), None)

found = devanagari_font()
if found is None:
    print("no Devanagari font registered - installing fonts-indic ...")
    subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-indic", "fonts-noto-core"],
                   check=False, capture_output=True)
    # matplotlib caches its font list on disk, and attention_analysis.py runs as a
    # separate process -- registering the font only in this one leaves the subprocess
    # reading a stale cache. Delete it so every later process rescans.
    for stale in pathlib.Path(matplotlib.get_cachedir()).glob("fontlist-*.json"):
        stale.unlink()
        print(f"  cleared stale font cache: {stale.name}")
    for path in font_manager.findSystemFonts(fontpaths=None, fontext="ttf"):
        try:
            font_manager.fontManager.addfont(path)
        except Exception:
            pass
    found = devanagari_font()

print("Devanagari font:", found or "NONE - heatmap axes will fall back to positions")

## 5. Sanity checks

Four things worth confirming before spending GPU time: the pretrained checkpoint loads
and is the one you think it is, its vocabulary matches the tokenizer, the reasoning
splits are the expected size, and no entity name is shared between train and test.

The last one is the leakage guarantee the assignment asks you to report. It was verified
when the data was generated; this re-checks it against the files actually uploaded.

In [ ]:
from lma.config import ModelConfig

for lang in LANGUAGES:
    d = LANG_DIRS[lang]
    print(f"{'='*70}\n{LABELS[lang]}\n{'='*70}")

    payload = torch.load(f"checkpoints/{d}/best.pt", map_location="cpu",
                         weights_only=False)
    cfg = ModelConfig(**payload["model_config"])
    print(f"  pretrained step {payload['step']:,}  "
          f"{payload['tokens_seen']:,} tokens seen  "
          f"val loss {payload['best_val_loss']:.4f}")
    print(f"  architecture    {cfg.n_layer}L x {cfg.n_head}H x {cfg.d_model}d  "
          f"vocab {cfg.vocab_size:,}")

    sp = sentencepiece.SentencePieceProcessor(
        model_file=f"{d}/tokenizer/{lang}.model")
    if sp.get_piece_size() != cfg.vocab_size:
        raise SystemExit(
            f"VOCABULARY MISMATCH: checkpoint expects {cfg.vocab_size:,}, "
            f"tokenizer has {sp.get_piece_size():,}. The embedding table would address "
            "token ids this tokenizer never emits.")
    print(f"  tokenizer       {sp.get_piece_size():,} pieces  (matches)")

    splits = {}
    for split in ("train", "validation", "test"):
        with open(f"{d}/reasoning/{split}.jsonl", encoding="utf-8") as fh:
            splits[split] = [json.loads(line) for line in fh]
        print(f"  {split:11s} {len(splits[split]):>6,} examples")

    names = {s: {e for r in rows for e in r["entities"]} for s, rows in splits.items()}
    overlap = names["train"] & names["test"]
    if overlap:
        raise SystemExit(f"LEAKAGE: {len(overlap)} entity names shared between train "
                         f"and test: {sorted(overlap)[:5]}")
    print(f"  leakage check   train {len(names['train'])} names, "
          f"test {len(names['test'])} names, 0 shared")
    print()

## 6. Finetuning

Roughly 20 minutes per language. The loss is masked to the answer span, so the objective
matches the reported metric; the unmasked loss is logged alongside it as a diagnostic.

`--resume` is passed, so re-running this cell after an interruption continues from the
last checkpoint rather than starting over.

In [ ]:
if RUN_FINETUNE:
    for lang in LANGUAGES:
        d = LANG_DIRS[lang]
        out = Path(f"/kaggle/working/checkpoints/{d}-finetuned")
        out.mkdir(parents=True, exist_ok=True)

        print(f"\n{'='*70}\n  finetuning {LABELS[lang]}\n{'='*70}")
        subprocess.run([
            sys.executable, "-m", "scripts.finetune", "--lang", lang,
            "--pretrained", f"checkpoints/{d}/best.pt",
            "--data-dir", f"{d}/reasoning",
            "--out-dir", str(out),
            "--epochs", str(EPOCHS),
            "--batch-size", str(BATCH_SIZE),
            "--micro-batch-size", str(MICRO_BATCH),
            "--learning-rate", str(LEARNING_RATE),
            "--max-len", str(MAX_LEN),
            "--warmup-steps", str(WARMUP_STEPS),
            "--eval-every", str(EVAL_EVERY),
            "--eval-batches", str(EVAL_BATCHES),
            "--resume",
        ], check=True)
else:
    print("finetuning skipped")

## 7. Reasoning evaluation

Generates an answer for every test prompt from **both** the pretrained and the finetuned
model, and scores exact match. Three numbers are reported:

- **strict** — the generated text equals the gold answer
- **lenient** — also accepts the oblique form (`डिब्बे` for `डिब्बा`), since a model that
  inflects the entity as the prompt did has still identified it correctly
- **first-word** — separates "cannot reason" from "will not stop", which dominates the
  pretrained baseline

A per-example chance baseline is reported too, so a pretrained model's noise is not
mistaken for partial competence.

In [ ]:
if RUN_EVAL:
    for lang in LANGUAGES:
        d = LANG_DIRS[lang]
        cmd = [
            sys.executable, "-m", "scripts.eval_reasoning", "--lang", lang,
            "--pretrained", f"checkpoints/{d}/best.pt",
            "--finetuned", f"/kaggle/working/checkpoints/{d}-finetuned/best.pt",
            "--data-dir", f"{d}/reasoning",
            "--split", "test",
            "--max-new-tokens", str(MAX_NEW),
        ]
        if EVAL_LIMIT:
            cmd += ["--limit", str(EVAL_LIMIT)]

        print(f"\n{'='*70}\n  evaluating {LABELS[lang]}\n{'='*70}")
        subprocess.run(cmd, check=True)
else:
    print("evaluation skipped")

## 8. Attention analysis, pretrained against finetuned

Three runs per language:

1. **Heatmaps on a comparative-reasoning prompt, pretrained** — the baseline
2. **The same prompt, finetuned** — directly comparable, same sentence, same layers
3. **Per-head statistics on the finetuned model**, over the validation split, so entropy
   and distance stay comparable with the Phase 2 numbers

Runs 1 and 2 pass `--heatmaps-only`, which skips the token stream and, importantly,
leaves `attention_stats.json` untouched. Run 3 does write it, so it is renamed to
`attention_stats_finetuned.json` immediately afterwards — otherwise it would overwrite
the pretrained statistics that Phase 2 produced under the same name.

Layers 0 and 6 are drawn: one early, one late, which is what the assignment asks for.

In [ ]:
if RUN_ATTENTION:
    for lang in LANGUAGES:
        d = LANG_DIRS[lang]
        finetuned = f"/kaggle/working/checkpoints/{d}-finetuned/best.pt"

        # A real multi-hop prompt from the test split, so the comparison is made on the
        # kind of input the assignment asks about rather than on generic prose.
        with open(f"{d}/reasoning/test.jsonl", encoding="utf-8") as fh:
            rows = [json.loads(line) for line in fh]
        example = next(r for r in rows if r["template"] == "chain_pair" and r["hops"] >= 2)
        prompt = example["prompt"].replace("\n", " ")
        print(f"\n{'='*70}\n  {LABELS[lang]}\n{'='*70}")
        print(f"  prompt: {prompt}")
        print(f"  gold:   {example['answer']}\n")

        for stage, ckpt in (("pretrained", f"checkpoints/{d}/best.pt"),
                            ("finetuned", finetuned)):
            subprocess.run([
                sys.executable, "-m", "scripts.attention_analysis", "--lang", lang,
                "--checkpoint", ckpt,
                "--sentence", prompt,
                "--layers", "0", "6",
                "--heads", "0", "1", "2", "3", "4", "5", "6",
                "--heatmaps-only",
                "--out-dir", f"report/{d}/figures-reasoning-{stage}",
            ], check=False)

        # Full per-head statistics on the finetuned model.
        subprocess.run([
            sys.executable, "-m", "scripts.attention_analysis", "--lang", lang,
            "--checkpoint", finetuned,
            "--num-batches", "20", "--batch-size", "4",
            "--heads", "0", "1", "2", "3", "4", "5", "6",
            "--out-dir", f"report/{d}/figures-finetuned",
        ], check=False)

        # attention_analysis always writes report/<lang>/attention_stats.json, whatever
        # --out-dir says. Rename it so it cannot be confused with, or overwrite, the
        # pretrained statistics carried over from Phase 2.
        stats = Path(f"report/{d}/attention_stats.json")
        if stats.exists():
            stats.rename(f"report/{d}/attention_stats_finetuned.json")
            print(f"  statistics -> report/{d}/attention_stats_finetuned.json")
else:
    print("attention analysis skipped")

## 9. Package the results

In [ ]:
import tarfile

out = Path("/kaggle/working/phase3-results.tar.gz")

with tarfile.open(out, "w:gz") as tar:
    for lang in LANGUAGES:
        d = LANG_DIRS[lang]
        ckpt_dir = Path(f"/kaggle/working/checkpoints/{d}-finetuned")
        if ckpt_dir.exists():
            for pattern in ("best.pt", "step-*.pt", "finetune_log.jsonl"):
                for f in sorted(ckpt_dir.glob(pattern)):
                    tar.add(f, arcname=f"checkpoints/{d}-finetuned/{f.name}")
        report = ROOT/"report"/d
        if report.exists():
            for f in sorted(report.rglob("*")):
                if f.is_file():
                    tar.add(f, arcname=f"report/{d}/{f.relative_to(report)}")

print(f"wrote {out}  ({out.stat().st_size/1e9:.2f} GB)\n")
print("contents:")
with tarfile.open(out) as tar:
    for m in tar.getmembers():
        print(f"  {m.size/1e6:9.2f} MB  {m.name}")

print("\nNEXT: download from the Output tab, extract at the repo root, then locally:")
print("  python -m scripts.plot_training --lang hi --log checkpoints/hindi-finetuned/finetune_log.jsonl")
print("  (and write report/phase3.md from the JSON in report/<lang>/)")